# Week 67, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w067d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
chunksize, extra = divmod(50, 3 * 4)
print(chunksize + (1 if extra else 0))

**Predict before you run:** what will this print?



In [ ]:
async def f():
    return 1
c = f()
print(type(c).__name__, bool(c))
c.close()

**Predict before you run:** what will this print?



In [ ]:
delays = [4, 1, 2]
print(sum(delays), max(delays))

**Predict before you run:** what will this print?



In [ ]:
import asyncio
print(issubclass(asyncio.CancelledError, Exception), [min(30, 2 ** i) for i in range(6)])

**Predict before you run:** what will this print?



In [ ]:
rates = {"fetch": 10 / 0.5, "parse": 2 / 0.1, "save": 1 / 0.08}
print(min(rates, key=rates.get), round(min(rates.values()), 1))

### Exercise: Lock orders, again

From Week 66: write `lock_plan(transfers)`, the order in which each `(source, destination)` transfer should take its two account locks (sorted, as a tuple), and `could_deadlock(transfers)`: True if two transfers in the list move money in **opposite** directions between the same two accounts, so that taking locks in source-first order could deadlock.

In [ ]:
def lock_plan(transfers):
    ___

def could_deadlock(transfers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_locks():\n    \"lock ordering (Week 66)\"\n    transfers = [(\"x\", \"y\"), (\"y\", \"z\"), (\"y\", \"x\")]\n    assert lock_plan(transfers) == [(\"x\", \"y\"), (\"y\", \"z\"), (\"x\", \"y\")]\n    assert could_deadlock(transfers) and not could_deadlock([(\"x\", \"y\"), (\"x\", \"y\"), (\"y\", \"z\")])\n    assert not could_deadlock([])")

### Exercise: The price of small chunks

From Monday: write `chunk_payloads(items, chunksize)`, the size in bytes of each chunk when pickled for sending to a worker, and `overhead_ratio(items, chunksize)`: the total of those sizes divided by the size of pickling the whole list at once, rounded to 2 places (0.0 for no items). Small chunks repeat pickle's per-message overhead many times.

In [ ]:
import pickle

def chunk_payloads(items, chunksize):
    ___

def overhead_ratio(items, chunksize):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_payloads():\n    \"what pickling costs (Monday)\"\n    items = list(range(1000))\n    assert len(chunk_payloads(items, 100)) == 10 and len(chunk_payloads(items, 1)) == 1000\n    assert overhead_ratio(items, 1) > overhead_ratio(items, 100) >= 1.0\n    assert overhead_ratio([], 5) == 0.0 and overhead_ratio(items, 1000) == 1.0")

### Exercise: Forge: an async job runner

Write `AsyncQueue` and `run_jobs` to the specification above.

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Waitable:
    "Anything tasks can wait on: the loop adds the waiting task to .waiters."
    def __init__(self):
        self.waiters = deque()
    def wait(self):
        return _Wait(self)

class _Wait:
    def __init__(self, target):
        self.target = target
    def __await__(self):
        yield ("wait", self.target)

class Task(Waitable):
    def __init__(self, coro):
        super().__init__()
        self.coro, self.done, self.result = coro, False, None
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def wake(self, waitable, count=None):
        "Move waiting tasks (all, or up to count) back to the ready queue."
        while waitable.waiters and (count is None or count > 0):
            self.ready.append(waitable.waiters.popleft())
            if count is not None:
                count -= 1
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.wake(task)
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

class AsyncQueue(Waitable):
    def __init__(self, loop):
        ___

def run_jobs(jobs, workers, timeout, max_attempts=3, base_delay=1):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_jobs():\n    \"workers, timeouts and retries\"\n    jobs = [{\"name\": \"a\", \"durations\": [2]}, {\"name\": \"b\", \"durations\": [5, 1]},\n            {\"name\": \"c\", \"durations\": [1], \"fails\": [1, 2]}, {\"name\": \"d\", \"durations\": [1], \"fails\": [1, 2, 3]}]\n    report, end, log = run_jobs(jobs, 2, timeout=3)\n    assert report == {\"a\": {\"status\": \"ok\", \"attempts\": 1, \"finished\": 2},\n                      \"b\": {\"status\": \"ok\", \"attempts\": 2, \"finished\": 5},\n                      \"c\": {\"status\": \"ok\", \"attempts\": 3, \"finished\": 8},\n                      \"d\": {\"status\": \"failed\", \"attempts\": 3, \"finished\": 9}}\n    assert end == 9\n    assert log[:4] == [(0, 0, \"a\", 1), (0, 1, \"b\", 1), (2, 0, \"c\", 1), (3, 1, \"d\", 1)] and len(log) == 9\n\ndef test_edges():\n    \"edge cases\"\n    assert run_jobs([], 3, timeout=5)[:2] == ({}, 0)\n    report, end, _ = run_jobs([{\"name\": \"x\", \"durations\": [1]}], 1, timeout=3)\n    assert report == {\"x\": {\"status\": \"ok\", \"attempts\": 1, \"finished\": 1}} and end == 1\n    slow = [{\"name\": \"s\", \"durations\": [10]}]\n    report, end, _ = run_jobs(slow, 1, timeout=2, max_attempts=2, base_delay=5)\n    assert report[\"s\"] == {\"status\": \"failed\", \"attempts\": 2, \"finished\": 9} and end == 9\n    many = [{\"name\": f\"j{i}\", \"durations\": [2]} for i in range(6)]\n    assert run_jobs(many, 3, timeout=5)[1] == 4")

> **Why the backoff runs in its own task.** The tempting way to retry is for the worker to `await Sleep(backoff)` itself and then try again. It works, but while that worker sleeps it does nothing else, so a few failing jobs can tie up the whole pool while healthy jobs wait in the queue. Handing the delay to a separate small task, which just sleeps and puts the job back, keeps every worker busy with whatever is ready now. It's the same idea as Thursday's rule about blocking: in an event loop, any task that waits should wait without holding something others need. Real job systems (Celery, RQ, cloud task queues) schedule retries for later in exactly this way.

### Exercise: Forge bonus: a bounded pipeline

Using the loop in the starter, write `BoundedQueue(loop, maxsize)`: the coroutine `put(item)` waits while the queue holds `maxsize` items (wait on a second `Waitable`, `space`), then adds the item and wakes one getter; the coroutine `get()` waits while it's empty, then takes the oldest item and wakes one waiting putter; `peak` records the most items it has held. Then `run_pipeline(items, stages, maxsize)`, where `stages` is a list of `(seconds_per_item, workers)`: queues join the stages (one before each stage and one after the last), each stage's workers repeatedly take an item, `Sleep` for the stage's time, and put it into the next queue, and a collector records `(item, loop.now)` as items leave the last stage. Feed every item into the first queue, then shut the stages down in order: put one `None` per worker into each stage's input queue and await that stage's workers before moving to the next; finally put a `None` for the collector. Return `(done, loop.now, peaks)`, where `peaks` lists each queue's peak (the sentinels count).

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Waitable:
    "Anything tasks can wait on: the loop adds the waiting task to .waiters."
    def __init__(self):
        self.waiters = deque()
    def wait(self):
        return _Wait(self)

class _Wait:
    def __init__(self, target):
        self.target = target
    def __await__(self):
        yield ("wait", self.target)

class Task(Waitable):
    def __init__(self, coro):
        super().__init__()
        self.coro, self.done, self.result = coro, False, None
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def wake(self, waitable, count=None):
        "Move waiting tasks (all, or up to count) back to the ready queue."
        while waitable.waiters and (count is None or count > 0):
            self.ready.append(waitable.waiters.popleft())
            if count is not None:
                count -= 1
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.wake(task)
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

class BoundedQueue(Waitable):
    def __init__(self, loop, maxsize):
        ___

def run_pipeline(items, stages, maxsize):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pipeline():\n    \"back-pressure and bottlenecks\"\n    done, end, peaks = run_pipeline(list(range(6)), [(1, 1), (3, 1)], maxsize=2)\n    assert done == [(0, 4), (1, 7), (2, 10), (3, 13), (4, 16), (5, 19)] and end == 19\n    assert peaks[1] == 2\n    done, end, _ = run_pipeline(list(range(6)), [(1, 1), (3, 3)], maxsize=2)\n    assert [t for _, t in done] == [4, 5, 6, 7, 8, 9] and end == 9\n    done, end, _ = run_pipeline(list(range(4)), [(2, 2), (1, 1)], maxsize=1)\n    assert done == [(0, 3), (1, 4), (2, 5), (3, 6)] and end == 6\n    assert run_pipeline([], [(1, 1)], maxsize=1)[:2] == ([], 0)\n\ndef test_bounded_queue():\n    \"a queue that makes producers wait\"\n    loop = Loop()\n    q = BoundedQueue(loop, 2)\n    log = []\n    async def producer():\n        for i in range(4):\n            await q.put(i)\n            log.append((\"put\", i, loop.now))\n    async def consumer():\n        await Sleep(5)\n        for _ in range(4):\n            item = await q.get()\n            log.append((\"got\", item, loop.now))\n            await Sleep(1)\n    async def main():\n        p = loop.create_task(producer())\n        c = loop.create_task(consumer())\n        await p\n        await c\n    loop.run_until_complete(main())\n    assert log[:2] == [(\"put\", 0, 0), (\"put\", 1, 0)] and (\"put\", 2, 5) in log and q.peak == 2")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [multiprocessing](w067d1.html), [asyncio basics](w067d2.html), [async I/O patterns](w067d3.html), [async pitfalls](w067d4.html) or [choosing a concurrency model](w067d5.html).

## Mixed review

## Forge: an async job runner

Write `run_jobs(jobs, workers, timeout, max_attempts=3, base_delay=1)` using the event loop in the starter. Each job is a dict with a `name`, a list of `durations` (attempt `k` takes `durations[k - 1]`, or the last duration if there are fewer) and optionally `fails`, a collection of attempt numbers that fail even though they finish in time.

- `workers` worker tasks take jobs from an `AsyncQueue` (write it: `put` wakes one waiter; `get` waits while empty), one at a time.
- Each attempt is logged as `(loop.now, worker number, job name, attempt number)` when it starts.
- An attempt longer than `timeout` **times out**: the worker spends `timeout` units on it and it fails. Otherwise the worker spends its duration, and it succeeds unless its attempt number is in `fails`.
- A failed job with attempts left is **re-queued after a backoff** of `base_delay * 2 ** (attempt - 1)`, by a separate task that sleeps and then puts it back, so the worker moves straight on to other jobs.
- A job that succeeds, or fails its last attempt, is recorded as `{"status": "ok" or "failed", "attempts": n, "finished": loop.now}`. When every job is recorded, put one `None` per worker so they all stop.

Return `(report, loop.now, log)`, where `report` maps job names to their records.

## Forge bonus: a bounded pipeline

## Weekly quiz

## Week 67 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can use Process and Pool, explain pickling and chunking, pass messages, and explain start methods. ([Day 1](w067d1.html))
- I can write coroutines, run them concurrently, drive them by hand, and build a small event loop. ([Day 2](w067d2.html))
- I can limit concurrency, use async queues, timeouts and retries, async context managers and async generators. ([Day 3](w067d3.html))
- I can avoid blocking the loop, missing awaits, lost tasks, swallowed cancellations and races across await. ([Day 4](w067d4.html))
- I can choose and combine threads, processes and asyncio, find bottlenecks and measure efficiency. ([Day 5](w067d5.html))
- I built an async job runner and a bounded pipeline. (Today)

Tomorrow is a **rest day**. Next week: **Extending Python**: the C API, `ctypes` and `cffi`, Cython, Rust extensions with PyO3, and when to extend and when not to.